## Texas and Georgia Auction.com

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from rapidfuzz import process, fuzz
import subprocess
import json
import os
import re

In [2]:
folder = Path('raw_data/auction_com')
excel_files = sorted(folder.glob('*.xlsx'))
if not excel_files:
    raise FileNotFoundError(f'No Excel files found in {folder}')
auction = pd.concat([pd.read_excel(f, dtype={'Tax ID': 'string', 'APN': 'string', 'Zip': 'string'}) for f in excel_files], ignore_index=True)
# Remove genuinely empty imported records before matching
before = len(auction)
auction = auction.dropna(how='all').reset_index(drop=True)
print('Empty rows removed:', before - len(auction))

/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style,

Empty rows removed: 4


In [3]:
print(f"Files combined: {len(excel_files)}")
print(f"Total rows: {len(auction)}")
print(f"Total columns: {len(auction.columns)}")

display(auction.head())

Files combined: 4
Total rows: 1957
Total columns: 19


,Listing Type,Property Address,Property City,County,State,Zip,Status,Auction Date,Property Type,Square Footage,Bedrooms,Baths,Lot Size,Year Built,Est. Market Value,Starting Bid,Est. Credit Bid,APN,Trustee Number
0,Foreclosure,4645 PIGEON ISLAND DR,DECATUR,DeKalb,GA,30035,SALE_PENDING,2026-11-03,Single family home,2100.0,4.0,2.5,0.32,2002.0,298307.0,NaN,NaN,<NA>,25-292627
1,Foreclosure,6 BELMONT PARK DR,NEWNAN,Coweta,GA,30263,SALE_PENDING,2026-11-03,Single family home,2206.0,4.0,2.5,0.25,2004.0,304998.0,NaN,NaN,<NA>,TS 10933174
2,Foreclosure,2222 FLORIDA AVE,SAVANNAH,Chatham,GA,31404,SALE_PENDING,2026-11-03,Single family home,1741.0,4.0,2.5,0.16,1990.0,271704.0,NaN,NaN,20058 34003,26-440455
3,Foreclosure,113 BARRINGTON CIRCLE,RINCON,Effingham,GA,31326,SALE_PENDING,2026-11-03,Single family home,1343.0,0.0,2.0,0.63,1998.0,281939.0,NaN,NaN,<NA>,26-452257
4,Foreclosure,3685 PEACHTREE RD NE UNIT 10,ATLANTA,Fulton,GA,30319,SALE_PENDING,2026-11-03,Single family home,3125.0,4.0,3.5,0.06,1984.0,774565.0,NaN,NaN,<NA>,26-419341


In [4]:
# Create a consistent address and preserve ZIP as a string
zip_text = auction['Zip'].astype('string').str.replace(r'\.0$', '', regex=True).fillna('')
auction['Last Name'] = (
    auction['Property Address'].astype('string').str.strip().str.title() + ', ' +
    auction['Property City'].astype('string').str.strip().str.title() + ', ' +
    auction['State'].astype('string').str.strip().str.upper() + ' ' + zip_text
)
auction['Last Name'] = auction['Last Name'].fillna('')

## Match Addresses

In [5]:
query = """
SELECT
    Id,
    Name,
    Auction_Status__c,
    Auction_Trustee_Lookup__r.Name,
    Auction_Listing_ID__c,
    Auction_Opening_Bid__c,
    Online_Bidding__c
FROM Lead
WHERE (
    Auction_Date__c = LAST_N_DAYS:60
    OR Auction_Date__c = NEXT_N_DAYS:30)
AND State_Lookup__r.Name IN ('Texas', 'Georgia')
"""

result = subprocess.run(
    [
        "sf", "data", "query",
        "--query", query,
        "--target-org", "production_org",
        "--json"
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0"}
)

if result.returncode != 0:
    print("Salesforce Error:")
    print(result.stderr)
    print(result.stdout[:1000])

else:
    # Remove ANSI color codes if any remain
    import re

    clean_output = re.sub(
        r'\x1b\[[0-9;]*m',
        '',
        result.stdout
    )

    data = json.loads(clean_output)

    sf_leads = pd.json_normalize(data['result']['records'])

    print(f"Successfully retrieved {len(sf_leads)} Salesforce records")
    display(sf_leads.head())

Successfully retrieved 7600 Salesforce records


,Id,Name,Auction_Status__c,Auction_Listing_ID__c,Auction_Opening_Bid__c,Online_Bidding__c,attributes.type,attributes.url,Auction_Trustee_Lookup__r.attributes.type,Auction_Trustee_Lookup__r.attributes.url,Auction_Trustee_Lookup__r.Name
0,00QQg00000nIBX3MAO,"209 Rosewood Drive, Brunswick, GA 31525",Killed,2183399,100755.64,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
1,00QQg00000nIBX4MAO,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,2182824,161200.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
2,00QQg00000nIBX5MAO,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,2182090,224360.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
3,00QQg00000nIBX6MAO,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,2181984,NaN,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
4,00QQg00000nIBX7MAO,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,2181983,83840.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com


In [6]:
#Rename sf columns 

sf_leads = sf_leads.rename(columns={
    'Name': 'Last Name',
    'Auction_Status__c': 'Auction Status',
    'Auction_Trustee_Lookup__r.Name': 'Auction Trustee',
    'Auction_Listing_ID__c': 'Auction Listing ID',
    'Auction_Opening_Bid__c': 'Auction Opening Bid',
    'Online_Bidding__c': "Online Bidding"
})

In [7]:
sf_leads[['Last Name', 'Auction Status', 'Auction Trustee',
          'Auction Listing ID', 'Auction Opening Bid']].head()

,Last Name,Auction Status,Auction Trustee,Auction Listing ID,Auction Opening Bid
0,"209 Rosewood Drive, Brunswick, GA 31525",Killed,Auction.com,2183399,100755.64
1,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,Auction.com,2182824,161200.00
2,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,Auction.com,2182090,224360.00
3,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,Auction.com,2181984,NaN
4,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,Auction.com,2181983,83840.00


In [8]:
# Match addresses: only accepted matches are used for Salesforce updates
threshold = 85

def normalize_name(value):
    if pd.isna(value):
        return ''
    return ' '.join(str(value).upper().strip().split())

sf_lookup_names = {}
for name in sf_leads['Last Name'].dropna().unique():
    key = normalize_name(name)
    if key:
        sf_lookup_names.setdefault(key, name)
sf_names = list(sf_lookup_names)
auction['Auction Last Name'] = auction['Last Name']

def fuzzy_match_name(name):
    key = normalize_name(name)
    if not key or not sf_names:
        return pd.Series([None, None])
    found = process.extractOne(key, sf_names, scorer=fuzz.ratio)
    if found is None:
        return pd.Series([None, None])
    match_name, score, _ = found
    return pd.Series([sf_lookup_names[match_name], round(score, 2)])

auction[['Salesforce Last Name', 'Match Confidence']] = auction['Auction Last Name'].apply(fuzzy_match_name)
accepted = auction['Match Confidence'].ge(threshold).fillna(False)
# Clear rejected suggestions so all downstream lookups use accepted matches only
auction.loc[~accepted, 'Salesforce Last Name'] = None
auction.loc[accepted, 'Last Name'] = auction.loc[accepted, 'Salesforce Last Name']
print('Accepted Salesforce matches:', int(accepted.sum()))

Accepted Salesforce matches: 770


In [9]:
#View matches

display(auction[
    ['Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Auction Last Name,Salesforce Last Name,Match Confidence
0,"4645 Pigeon Island Dr, Decatur, GA 30035",None,63.16
1,"6 Belmont Park Dr, Newnan, GA 30263",None,63.89
2,"2222 Florida Ave, Savannah, GA 31404",None,61.97
3,"113 Barrington Circle, Rincon, GA 31326",None,66.67
4,"3685 Peachtree Rd Ne Unit 10, Atlanta, GA 30319",None,68.75
...,...,...,...
1952,"238 Saratoga Drive, San Antonio, TX 78213","238 Saratoga Dr, San Antonio, TX 78213",96.20
1953,"17116 Border Lake Lane, Houston, TX 77044","17116 Border Lake Ln, Houston, TX 77044",97.50
1954,"9846 Spruce Ridge Drive, Converse, TX 78109","9846 Spruce Ridge Dr, Converse, TX 78109",96.39
1955,"13151 Celtic, Horizon City, TX 79928",None,78.48


In [10]:
#Look at differences

#Common street suffixes
suffix_map = {
    'DRIVE': 'DR',
    'STREET': 'ST',
    'AVENUE': 'AVE',
    'ROAD': 'RD',
    'LANE': 'LN',
    'BOULEVARD': 'BLVD',
    'COURT': 'CT',
    'CIRCLE': 'CIR',
    'PLACE': 'PL',
    'TERRACE': 'TER',
    'TRAIL': 'TRL',
    'PARKWAY': 'PKWY',
    'HIGHWAY': 'HWY',
    'WAY': 'WAY'
}

def get_address_parts(address):
    if pd.isna(address):
        return '', ''

    address = str(address).strip()
    parts = [p.strip() for p in address.split(',')]

    # Expected format: Street, City, TX ZIP
    street = parts[0] if len(parts) >= 1 else ''
    city = parts[1] if len(parts) >= 3 else ''

    return street.upper(), city.upper()

def normalize_street(street):
    street = re.sub(r'[^\w\s]', '', street.upper())
    words = street.split()
    return ' '.join(suffix_map.get(word, word) for word in words)

def compare_addresses(row):
    auction_street, auction_city = get_address_parts(row['Auction Last Name'])
    sf_street, sf_city = get_address_parts(row['Salesforce Last Name'])

    # Detect abbreviation-only differences
    suffix_difference = (
        auction_street != sf_street
        and normalize_street(auction_street) == normalize_street(sf_street)
    )

    city_difference = auction_city != sf_city

    return pd.Series({
        'Street Suffix Difference': suffix_difference,
        'City Difference': city_difference,
        'Auction City': auction_city,
        'Salesforce City': sf_city
    })

# Compare matched records
address_comparison = auction[
    (auction['Match Confidence'] >= 85) &
    auction['Salesforce Last Name'].notna()
].copy()

address_comparison = address_comparison.join(
    address_comparison.apply(compare_addresses, axis=1)
)

In [11]:
differences = address_comparison[
    address_comparison['Street Suffix Difference'] |
    address_comparison['City Difference']
]

display(differences[
    [
        'Last Name',
        'Auction Last Name',
        'Salesforce Last Name',
        'Match Confidence',
        'Street Suffix Difference',
        'City Difference'
    ]
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence,Street Suffix Difference,City Difference
783,"1914 Cross Draw Trl, Leander, TX 78641","1914 Cross Draw Trail, Leander, TX 78641","1914 Cross Draw Trl, Leander, TX 78641",97.44,True,False
788,"4104 Dartmoor Dr, Aubrey, TX 76227","4104 Dartmoor Drive, Aubrey, TX 76227","4104 Dartmoor Dr, Aubrey, TX 76227",95.77,True,False
790,"1005 Hyde Park Dr, Mckinney, TX 75069","1005 Hyde Park Drive, Mckinney, TX 75069","1005 Hyde Park Dr, Mckinney, TX 75069",96.10,True,False
793,"1554 Serenade Ln, Dallas, TX 75241","1554 Serenade Lane, Dallas, TX 75241","1554 Serenade Ln, Dallas, TX 75241",97.14,True,False
797,"7304 Glacier Dr, El Paso, TX 79911","7304 Glacier Dr., El Paso, TX 79911","7304 Glacier Dr, El Paso, TX 79911",98.55,True,False
...,...,...,...,...,...,...
1922,"4615 Wadsworth Dr, Dallas, TX 75216","4615 Wadsworth Drive, Dallas, TX 75216","4615 Wadsworth Dr, Dallas, TX 75216",95.89,True,False
1923,"8019 Oregano Dr, Baytown, TX 77521","8019 Oregano Drive, Baytown, TX 77521","8019 Oregano Dr, Baytown, TX 77521",95.77,True,False
1952,"238 Saratoga Dr, San Antonio, TX 78213","238 Saratoga Drive, San Antonio, TX 78213","238 Saratoga Dr, San Antonio, TX 78213",96.20,True,False
1953,"17116 Border Lake Ln, Houston, TX 77044","17116 Border Lake Lane, Houston, TX 77044","17116 Border Lake Ln, Houston, TX 77044",97.50,True,False


In [12]:
print("Street suffix differences:",
      address_comparison['Street Suffix Difference'].sum())

print("City differences:",
      address_comparison['City Difference'].sum())

print("Both:",
      (
          address_comparison['Street Suffix Difference'] &
          address_comparison['City Difference']
      ).sum())

Street suffix differences: 184
City differences: 4
Both: 1


In [13]:
low_confidence = auction[
    (auction['Match Confidence'] < 85) |
    (auction['Match Confidence'].isna())
]

display(low_confidence[
    ['Last Name','Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence
0,"4645 Pigeon Island Dr, Decatur, GA 30035","4645 Pigeon Island Dr, Decatur, GA 30035",None,63.16
1,"6 Belmont Park Dr, Newnan, GA 30263","6 Belmont Park Dr, Newnan, GA 30263",None,63.89
2,"2222 Florida Ave, Savannah, GA 31404","2222 Florida Ave, Savannah, GA 31404",None,61.97
3,"113 Barrington Circle, Rincon, GA 31326","113 Barrington Circle, Rincon, GA 31326",None,66.67
4,"3685 Peachtree Rd Ne Unit 10, Atlanta, GA 30319","3685 Peachtree Rd Ne Unit 10, Atlanta, GA 30319",None,68.75
...,...,...,...,...
1945,"5030 Aspen Orchard Drive, Houston, TX 77066","5030 Aspen Orchard Drive, Houston, TX 77066",None,77.92
1947,"27908 Parkside Creek Drive, Spring, TX 77386","27908 Parkside Creek Drive, Spring, TX 77386",None,78.05
1950,"412 Carver St, Lockhart, TX 78644","412 Carver St, Lockhart, TX 78644",None,78.26
1955,"13151 Celtic, Horizon City, TX 79928","13151 Celtic, Horizon City, TX 79928",None,78.48


## Format Data

In [14]:
#Online Bidding
auction['Online Bidding'] = 'Yes'

#Auction Trustee
auction['Auction Trustee (Lookup)'] = 'Auction.com'

In [15]:
# Standard county normalization for BOTH Market and Lead Manager lookup
texascounties = pd.read_csv('sf_data/texascounties.csv', dtype=str)
georgiacounties = pd.read_csv('sf_data/georgiacounties.csv', dtype=str)

def normalize_county(value):
    if pd.isna(value):
        return ''
    value = str(value).strip().upper()
    value = re.sub(r'\s*-\s*(TX|GA)$', '', value)
    value = re.sub(r'\s+COUNTY$', '', value)
    return re.sub(r'[^A-Z0-9]', '', value)  # De Kalb, DeKalb, Dekalb all match

# Original county is retained in oldCounty, output county in County
if 'oldCounty' not in auction.columns:
    auction = auction.rename(columns={'County': 'oldCounty'})
auction['State'] = auction['State'].astype('string').str.strip().str.upper()
auction['Normalized County'] = auction['oldCounty'].map(normalize_county)
auction.loc[auction['Normalized County'].eq('DEKALB'), 'oldCounty'] = 'Dekalb'

county_lookup = {}
for state, df in [('TX', texascounties), ('GA', georgiacounties)]:
    for _, r in df.iterrows():
        key = (state, normalize_county(r['County']))
        if key[1]:
            county_lookup[key] = '' if pd.isna(r['Market']) else str(r['Market']).strip()

auction['Market'] = [county_lookup.get((s, county), '') for s, county in zip(auction['State'], auction['Normalized County'])]
auction['County'] = auction['oldCounty'].astype('string').str.strip().str.title() + ' - ' + auction['State']
auction['Decision'] = 'Add New'
auction.loc[auction['Salesforce Last Name'].notna(), 'Decision'] = 'Update'
auction.loc[auction['Market'].str.casefold().eq('exclude'), 'Decision'] = 'Exclude'
missing_markets = auction[auction['Market'].eq('')]
print('Total auctions:', len(auction))
print('Missing Markets:', len(missing_markets))
if len(missing_markets):
    display(missing_markets[['Last Name','State','oldCounty','Market']])

Total auctions: 1957
Missing Markets: 4


,Last Name,State,oldCounty,Market
633,,<NA>,NaN,
655,,<NA>,NaN,
766,,<NA>,NaN,
1956,,<NA>,NaN,


## Lead Manager

In [16]:
# One authoritative Lead Manager calculation; FIRST IMPORT uses TX Online manager
sf_record_lookup = sf_leads.drop_duplicates(subset='Last Name').set_index('Last Name')
auction['Salesforce Auction Trustee'] = auction['Salesforce Last Name'].map(sf_record_lookup['Auction Trustee'])
auction['Salesforce Online Bidding'] = auction['Salesforce Last Name'].map(sf_record_lookup['Online Bidding'])

def get_col(df, name):
    # Accept the current Georgia CSV name, with legacy name as fallback
    if name in df.columns:
        return name
    if name == 'Lead Manager Name' and 'Lead Manager' in df.columns:
        return 'Lead Manager'
    raise KeyError(f'Missing {name!r}; available columns: {list(df.columns)}')

manager_county_lookup = {}
for _, r in texascounties.iterrows():
    manager_county_lookup[('TX', normalize_county(r['County']))] = r['Online Lead Manager']
ga_manager_col = get_col(georgiacounties, 'Lead Manager Name')
for _, r in georgiacounties.iterrows():
    manager_county_lookup[('GA', normalize_county(r['County']))] = r[ga_manager_col]

auction['Lead Manager Name'] = [manager_county_lookup.get((s, county), '') for s, county in zip(auction['State'], auction['Normalized County'])]
auction['Lead Manager Name'] = auction['Lead Manager Name'].fillna('').astype(str).str.strip()
leadmanagers = pd.read_csv('sf_data/leadmanagers.csv', dtype=str)
leadmanagers['Lead Manager'] = leadmanagers['Lead Manager'].fillna('').str.strip()
manager_lookup = leadmanagers.drop_duplicates('Lead Manager').set_index('Lead Manager')['Salesforce ID']
auction['Lead Manager'] = auction['Lead Manager Name'].map(manager_lookup).fillna('')
excluded = auction['Market'].str.casefold().eq('exclude')
auction.loc[excluded, ['Lead Manager Name','Lead Manager']] = ''
missing_managers = auction.loc[~excluded & (auction['Lead Manager Name'].eq('') | auction['Lead Manager'].eq(''))]
print('Missing Lead Managers (non-excluded):', len(missing_managers))
if len(missing_managers):
    display(missing_managers[['Last Name','State','oldCounty','Market','Lead Manager Name','Lead Manager']])

Missing Lead Managers (non-excluded): 4


,Last Name,State,oldCounty,Market,Lead Manager Name,Lead Manager
633,,<NA>,NaN,,,
655,,<NA>,NaN,,,
766,,<NA>,NaN,,,
1956,,<NA>,NaN,,,


In [17]:
# Lead Manager is assigned in the preceding cell.
# Do not recalculate or overwrite it here.
print('Assigned Lead Manager IDs:', auction['Lead Manager'].ne('').sum())

Assigned Lead Manager IDs: 1527


In [18]:
display(auction[
    [
        'Last Name',
        'Salesforce Last Name',
        'Salesforce Auction Trustee',
        'Salesforce Online Bidding',
        'Lead Manager Name',
        'Lead Manager'
    ]
].head(20))

,Last Name,Salesforce Last Name,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager
0,"4645 Pigeon Island Dr, Decatur, GA 30035",None,NaN,NaN,,
1,"6 Belmont Park Dr, Newnan, GA 30263",None,NaN,NaN,,
2,"2222 Florida Ave, Savannah, GA 31404",None,NaN,NaN,,
3,"113 Barrington Circle, Rincon, GA 31326",None,NaN,NaN,,
4,"3685 Peachtree Rd Ne Unit 10, Atlanta, GA 30319",None,NaN,NaN,,
5,"101B Oak Hill Drive, Calhoun, GA 30701",None,NaN,NaN,David Riehle,0051H00000ASDES
6,"140 Valley Road, Stockbridge, GA 30281",None,NaN,NaN,,
7,"185 Rock Bridge Dr, Ringgold, GA 30736",None,NaN,NaN,David Riehle,0051H00000ASDES
8,"3520 Howard Ave, Columbus, GA 31904",None,NaN,NaN,,
9,"3378 Shoals Ridge Drive, Dacula, GA 30019",None,NaN,NaN,,


In [19]:
#No lead manager

display(auction[
    auction['Lead Manager Name'].isna() |
    (auction['Lead Manager Name'].astype(str).str.strip() == '')
])

,Listing Type,Property Address,Property City,oldCounty,State,Zip,Status,Auction Date,Property Type,Square Footage,...,Online Bidding,Auction Trustee (Lookup),Normalized County,Market,County,Decision,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager
0,Foreclosure,4645 PIGEON ISLAND DR,DECATUR,Dekalb,GA,30035,SALE_PENDING,2026-11-03,Single family home,2100.0,...,Yes,Auction.com,DEKALB,Exclude,Dekalb - GA,Exclude,NaN,NaN,,
1,Foreclosure,6 BELMONT PARK DR,NEWNAN,Coweta,GA,30263,SALE_PENDING,2026-11-03,Single family home,2206.0,...,Yes,Auction.com,COWETA,Exclude,Coweta - GA,Exclude,NaN,NaN,,
2,Foreclosure,2222 FLORIDA AVE,SAVANNAH,Chatham,GA,31404,SALE_PENDING,2026-11-03,Single family home,1741.0,...,Yes,Auction.com,CHATHAM,Exclude,Chatham - GA,Exclude,NaN,NaN,,
3,Foreclosure,113 BARRINGTON CIRCLE,RINCON,Effingham,GA,31326,SALE_PENDING,2026-11-03,Single family home,1343.0,...,Yes,Auction.com,EFFINGHAM,Exclude,Effingham - GA,Exclude,NaN,NaN,,
4,Foreclosure,3685 PEACHTREE RD NE UNIT 10,ATLANTA,Fulton,GA,30319,SALE_PENDING,2026-11-03,Single family home,3125.0,...,Yes,Auction.com,FULTON,Exclude,Fulton - GA,Exclude,NaN,NaN,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1924,Foreclosure,27902 CHIAPAS DR,SPRING,Montgomery,TX,77386,SALE_PENDING,2026-11-03,Single family home,3163.0,...,Yes,Auction.com,MONTGOMERY,Exclude,Montgomery - TX,Exclude,NaN,NaN,,
1933,Foreclosure,14822 ARCHER CT,MONTGOMERY,Montgomery,TX,77316,SALE_PENDING,2026-11-03,Single family home,1388.0,...,Yes,Auction.com,MONTGOMERY,Exclude,Montgomery - TX,Exclude,NaN,NaN,,
1943,Foreclosure,818 STRANGE DR,RICHMOND,Fort Bend,TX,77406,CANCELED,2026-11-03,Single family home,1440.0,...,Yes,Auction.com,FORTBEND,Exclude,Fort Bend - TX,Exclude,NaN,NaN,,
1947,Foreclosure,27908 PARKSIDE CREEK DRIVE,SPRING,Montgomery,TX,77386,SALE_PENDING,2026-11-03,Single family home,2513.0,...,Yes,Auction.com,MONTGOMERY,Exclude,Montgomery - TX,Exclude,NaN,NaN,,


In [20]:
auction[['oldCounty', 'County', 'Market', 'Lead Manager Name']].head()

,oldCounty,County,Market,Lead Manager Name
0,Dekalb,Dekalb - GA,Exclude,
1,Coweta,Coweta - GA,Exclude,
2,Chatham,Chatham - GA,Exclude,
3,Effingham,Effingham - GA,Exclude,
4,Fulton,Fulton - GA,Exclude,


In [21]:
display(auction.loc[
    auction['Market'] == 'Exclude',
    ['Last Name', 'State', 'County', 'Market', 'Lead Manager Name', 'Decision']
])

,Last Name,State,County,Market,Lead Manager Name,Decision
0,"4645 Pigeon Island Dr, Decatur, GA 30035",GA,Dekalb - GA,Exclude,,Exclude
1,"6 Belmont Park Dr, Newnan, GA 30263",GA,Coweta - GA,Exclude,,Exclude
2,"2222 Florida Ave, Savannah, GA 31404",GA,Chatham - GA,Exclude,,Exclude
3,"113 Barrington Circle, Rincon, GA 31326",GA,Effingham - GA,Exclude,,Exclude
4,"3685 Peachtree Rd Ne Unit 10, Atlanta, GA 30319",GA,Fulton - GA,Exclude,,Exclude
...,...,...,...,...,...,...
1920,"1718 Regal Blue Ct, Fresno, TX 77545",TX,Fort Bend - TX,Exclude,,Exclude
1924,"27902 Chiapas Dr, Spring, TX 77386",TX,Montgomery - TX,Exclude,,Exclude
1933,"14822 Archer Ct, Montgomery, TX 77316",TX,Montgomery - TX,Exclude,,Exclude
1943,"818 Strange Dr, Richmond, TX 77406",TX,Fort Bend - TX,Exclude,,Exclude


In [22]:
# Remove duplicate properties once, and report the number removed
before = len(auction)
auction = auction.drop_duplicates(subset=['Last Name'], keep='first').reset_index(drop=True)
print('Duplicates removed:', before - len(auction))
print('Remaining properties:', len(auction))

Duplicates removed: 8
Remaining properties: 1949


In [23]:
# Duplicate removal already completed above; no second mutation.
print('Remaining properties:', len(auction))

Remaining properties: 1949


## Split Data

In [24]:
export_data = Path('export_data')

# Export records marked Update
auction[auction['Decision'] == 'Update'].to_csv(
    export_data / 'update_auction.csv',
    index=False
)

# Export records marked Add New
auction[auction['Decision'] == 'Add New'].to_csv(
    export_data / 'create_auction.csv',
    index=False
)

# Confirm exported records
print("Update Auction:", (auction['Decision'] == 'Update').sum())
print("Create Auction:", (auction['Decision'] == 'Add New').sum())
print("Excluded:", (auction['Decision'] == 'Exclude').sum())

Update Auction: 765
Create Auction: 758
Excluded: 426


## Format update_auction

In [25]:
update_auction = auction[auction['Decision'] == 'Update'].copy()

#Add OB from Starting Bid
update_auction['Auction Opening Bid'] = (
    update_auction['Starting Bid'].replace(r'^\s*$', pd.NA, regex=True)
    .fillna('')
)

In [26]:
# Salesforce status lookup (deduplicate names before mapping)
sf_status_lookup = sf_leads.drop_duplicates('Last Name').set_index('Last Name')['Auction Status']
update_auction['Salesforce Auction Status'] = update_auction['Salesforce Last Name'].map(sf_status_lookup)

Priority,	Condition,	                                Auction Status

1	        Original Status = CANCELED,	                Cancelled

2	        Salesforce Auction Status = Killed,	        Killed

3	        Salesforce Auction Status = Clear to Bid,	Clear to Bid

4	        Auction Opening Bid has a value,	         Active

5	        Everything else,	                         Pending

In [27]:
# Default status
update_auction['Auction Status'] = 'Pending'

# Active if Opening Bid exists
has_bid = (
    update_auction['Auction Opening Bid'].notna() &
    update_auction['Auction Opening Bid'].astype(str).str.strip().ne('')
)

update_auction.loc[has_bid, 'Auction Status'] = 'Active'

# Preserve Salesforce Clear to Bid
#update_auction.loc[
    #update_auction['Salesforce Auction Status'] == 'Clear to Bid',
    #'Auction Status'
#] = 'Clear to Bid'

#Property type
update_auction.loc[
    update_auction['Property Type'] != 'Single family home',
    'Auction Status'
] = 'Killed'

# Canceled auction takes highest priority
update_auction.loc[
    update_auction['Status'] == 'CANCELED',
    'Auction Status'
] = 'Cancelled'

In [28]:
display(update_auction[
    [
        'Last Name',
        'Decision',
        'Status',
        'Salesforce Auction Status',
        'Starting Bid',
        'Auction Opening Bid',
        'Auction Status'
    ]
].head(20))

print(update_auction['Auction Status'].value_counts())


,Last Name,Decision,Status,Salesforce Auction Status,Starting Bid,Auction Opening Bid,Auction Status
39,"1105 Coffee Ave S, Douglas, GA 31533",Update,SALE_PENDING,Cancelled,NaN,,Pending
102,"4348 Newland St, Hephzibah, GA 30815",Update,SALE_PENDING,Killed,NaN,,Pending
118,"380 Paul Smith Road, Covington, GA 30014",Update,SALE_PENDING,Cancelled,NaN,,Pending
189,"3581 Morinda Drive, Douglasville, GA 30135",Update,SALE_PENDING,Cancelled,NaN,,Pending
196,"220 Deer Trl, Fayetteville, GA 30214",Update,SALE_PENDING,Killed,NaN,,Pending
314,"280 Whippoorwill Way Ne, Ludowici, GA 31316",Update,SALE_PENDING,Killed,NaN,,Pending
469,"1121 Highway 314, Fayetteville, GA 30214",Update,SALE_PENDING,Pending,NaN,,Killed
483,"8391 Kennedy Bridge Road, Register, GA 30452",Update,SALE_PENDING,Killed,NaN,,Pending
587,"315 Hidden Creek Cir, Warner Robins, GA 31088",Update,SALE_PENDING,Cancelled,NaN,,Pending
634,"16304 Pecan St, Channelview, TX 77530",Update,SALE_PENDING,Pending,NaN,,Pending


Auction Status
Pending      550
Cancelled    153
Killed        32
Active        30
Name: count, dtype: int64


In [29]:
# Add Killed Reason
update_auction['Killed Reason'] = update_auction['Auction Status'].apply(
    lambda x: 'Property Type' if x == 'Killed' else ''
)

In [30]:
display(update_auction.loc[
    (update_auction['Auction Status'] == 'Killed') &
    (update_auction['Property Type'] != 'Single family home')
])

,Listing Type,Property Address,Property City,oldCounty,State,Zip,Status,Auction Date,Property Type,Square Footage,...,County,Decision,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager,Auction Opening Bid,Salesforce Auction Status,Auction Status,Killed Reason
469,Foreclosure,909 HIGHWAY 314,FAYETTEVILLE,Fayette,GA,30214,SALE_PENDING,2026-11-03,Unknown,NaN,...,Fayette - GA,Update,Auction.com,None,David Riehle,0051H00000ASDES,,Pending,Killed,Property Type
681,Foreclosure,1919 CR 2130,"TELEPHONE,",Fannin,TX,75488,SALE_PENDING,2026-11-03,Unknown,NaN,...,Fannin - TX,Update,Auction.com,Yes,Kevin Rapport,005Qg00000VKIwb,,Cancelled,Killed,Property Type
778,Foreclosure,5913 LAKE HUBBARD PKWY 121,GARLAND,Dallas,TX,75043,SALE_PENDING,2026-11-03,Condominium unit,920.0,...,Dallas - TX,Update,Lookup Needed,None,Ernie Loreto,005Qg00000UWrpV,,Killed,Killed,Property Type
783,Foreclosure,2910 ACOPIO BEND UNIT 61,AUSTIN,Travis,TX,78745,SALE_PENDING,2026-11-03,Condominium unit,1458.0,...,Travis - TX,Update,Lookup Needed,None,Aaron Amuchastegui,005Qg00000Voakn,,Killed,Killed,Property Type
811,Foreclosure,20730 DAPPLED RIDGE WAY,HUMBLE,Harris,TX,77338,SALE_PENDING,2026-11-03,Single family attached,4414.0,...,Harris - TX,Update,Lookup Needed,None,Michael Shiner,005Qg00000Nsy1x,,Pending,Killed,Property Type
830,Foreclosure,5025 MISSION AVE,DALLAS,Dallas,TX,75206,SALE_PENDING,2026-11-03,Duplex building,2814.0,...,Dallas - TX,Update,Lookup Needed,None,Ernie Loreto,005Qg00000UWrpV,,Killed,Killed,Property Type
855,Foreclosure,14333 PRESTON ROAD,DALLAS,Dallas,TX,75254,SALE_PENDING,2026-11-03,Condominium unit,1028.0,...,Dallas - TX,Update,Lookup Needed,None,Ernie Loreto,005Qg00000UWrpV,,Killed,Killed,Property Type
888,Foreclosure,5102 CHENEVERT,HOUSTON,Harris,TX,77004,SALE_PENDING,2026-11-03,Duplex building,2472.0,...,Harris - TX,Update,Auction.com,Yes,Michael Shiner,005Qg00000Nsy1x,,Pending,Killed,Property Type
985,Foreclosure,701 CIRCLE WAY,JARRELL,Williamson,TX,76537,SALE_PENDING,2026-11-03,Condominium unit,2520.0,...,Williamson - TX,Update,Lookup Needed,None,Aaron Amuchastegui,005Qg00000Voakn,,Killed,Killed,Property Type
1001,Foreclosure,8518 CAHILL DR #7,AUSTIN,Williamson,TX,78729,SALE_PENDING,2026-11-03,Condominium unit,1333.0,...,Williamson - TX,Update,Lookup Needed,None,Aaron Amuchastegui,005Qg00000Voakn,,Killed,Killed,Property Type


In [31]:
#FIRST IMPORT ONLY

#Format Auction Date Updated as MM/DD/YYYY
update_auction['Auction Date Updated'] = (
    pd.to_datetime(update_auction['Auction Date'], errors='coerce')
    .dt.strftime('%m/%d/%Y')
    .fillna('')
)

update_auction = update_auction.rename(columns={
    'Status': 'notStatus',
    'Year Built': 'notYearBuilt',
    'Baths': 'notBaths',
    'Trustee Number': 'NS Trustee Sale Number',
    'Market': 'notMarket',
    'County':'notCounty'
})

In [32]:
#ADDITIONAL IMPORTS
#Rename columns

#update_auction = update_auction.rename(columns={
    #'Status': 'notStatus',
    #'Year Built': 'notYearBuilt',
    #'Baths': 'notBaths',
    #'Trustee Number': 'NS Trustee Sale Number',
    #'Market': 'notMarket',
    #'County':'notCounty'
    #'Lead Manager': 'notLeadManager'
#})

In [33]:
update_auction.to_csv(
    export_data / 'update_auction.csv',
    index=False
)

## Format create_auction

In [34]:
create_auction = auction[auction['Decision'] == 'Add New'].copy()

In [35]:
#Format Tax ID

create_auction['Tax ID'] = (
    create_auction['APN']
    .astype('string')
    .replace(r'(?i)^\s*(null|nan|none)?\s*$', '', regex=True)
    .fillna('')
)

create_auction = create_auction.rename(columns={
    'APN': 'notAPN'
})

In [36]:
# Unique APN based on Market + address + ZIP
# Unmapped markets must be reviewed rather than silently creating a blank APN prefix.
market_map = {
    'TX Small County': 'TXSMALLCOUNTY', 'Austin': 'AUSTIN',
    'Waco': 'WACO', 'Dallas': 'DAL', 'San Antonio': 'SAN_ANT',
    'Houston': 'HOUS'
}
# Preserve existing prefixes; normalize other market names deterministically
market_prefix = create_auction['Market'].map(market_map)
market_prefix = market_prefix.fillna(create_auction['Market'].astype('string').str.upper().str.replace(r'[^A-Z0-9]+', '_', regex=True).str.strip('_'))
zip_part = create_auction['Zip'].astype('string').str.replace(r'\.0$', '', regex=True)
create_auction['APN'] = (market_prefix + '_' + create_auction['Property Address'].astype('string').str.strip().str.replace(r'\s+', '_', regex=True) + '_' + zip_part).fillna('')

In [37]:
# Create fields and auction status consistently with update dataset
create_auction['Type of Property'] = create_auction['Property Type'].replace({'Single family home': 'SFR'})
create_auction['Auction Opening Bid'] = create_auction['Starting Bid'].replace(r'^\s*$', pd.NA, regex=True).fillna('')
create_auction['Auction Status'] = 'Pending'
create_auction.loc[create_auction['Auction Opening Bid'].astype(str).str.strip().ne(''), 'Auction Status'] = 'Active'
create_auction.loc[create_auction['Type of Property'] != 'SFR', 'Auction Status'] = 'Killed'
create_auction.loc[create_auction['Status'].astype(str).str.upper().eq('CANCELED'), 'Auction Status'] = 'Cancelled'
create_auction['Killed Reason'] = create_auction['Auction Status'].apply(lambda x: 'Property Type' if x == 'Killed' else '')
create_auction['Auction Date Updated'] = pd.to_datetime(create_auction['Auction Date'], errors='coerce').dt.strftime('%m/%d/%Y').fillna('')
create_auction = create_auction.rename(columns={
    'Est. Market Value': 'RPR', 'Bedrooms': 'Beds',
    'Trustee Number': 'NS Trustee Sale Number',
    ' Square Footage': 'Bldg Sq Ft', 'Status': 'notStatus'
})

In [38]:
# Remove trailing .0 from numeric columns
for col in ['Auction Opening Bid', 'RPR', 'Beds', 'Bldg Sq Ft', 'Baths']:
    create_auction[col] = (
        create_auction[col]
        .astype('string')
        .str.replace(r'\.0$', '', regex=True)
        .fillna('')
    )

In [39]:
display(create_auction[
    [
        'Last Name',
        'Market',
        'County',
        'State',
        'APN',
        'Tax ID',
        'notAPN',
        'Type of Property',
        'Auction Opening Bid',
        'Auction Status',
        'Auction Date Updated',
        'RPR',
        'Beds',
        'Baths',
        'NS Trustee Sale Number',
        'Bldg Sq Ft',
        'notStatus'
    ]
].head(20))

print(create_auction['Auction Status'].value_counts())

,Last Name,Market,County,State,APN,Tax ID,notAPN,Type of Property,Auction Opening Bid,Auction Status,Auction Date Updated,RPR,Beds,Baths,NS Trustee Sale Number,Bldg Sq Ft,notStatus
5,"101B Oak Hill Drive, Calhoun, GA 30701",West Georgia,Gordon - GA,GA,WEST_GEORGIA_101B_OAK_HILL_DRIVE_30701,,<NA>,SFR,,Pending,11/03/2026,217478,3,2,26-24853GA,1216,SALE_PENDING
7,"185 Rock Bridge Dr, Ringgold, GA 30736",West Georgia,Catoosa - GA,GA,WEST_GEORGIA_185_ROCK_BRIDGE_DR_30736,,<NA>,SFR,,Pending,11/03/2026,330059,3,2,26-456152,1351,SALE_PENDING
10,"272 Grand Magnolia Street, Jackson, GA 30233",East Georgia,Butts - GA,GA,EAST_GEORGIA_272_GRAND_MAGNOLIA_STREET_30233,0033C-064-000.,0033C-064-000.,SFR,,Pending,11/03/2026,245657,3,4,24-7525,1778,SALE_PENDING
14,"1145 Corey Creek Dr, Monroe, GA 30655",East Georgia,Walton - GA,GA,EAST_GEORGIA_1145_COREY_CREEK_DR_30655,N166E069,N166E069,SFR,,Cancelled,11/03/2026,326741,0,3,26-24589GA,1899,CANCELED
16,"2840 Ga Hwy 127, Kathleen, GA 31047",West Georgia,Houston - GA,GA,WEST_GEORGIA_2840_GA_HWY_127_31047,00128B031000,00128B031000,SFR,,Pending,11/03/2026,,3,1,26-26948-FC01,1202,SALE_PENDING
18,"6982 Knollwood Dr, Morrow, GA 30260",West Georgia,Clayton - GA,GA,WEST_GEORGIA_6982_KNOLLWOOD_DR_30260,,<NA>,SFR,,Pending,11/03/2026,206457,3,2,TS 10953909,1211,SALE_PENDING
19,"101 Jessica Dr, Warner Robins, GA 31093",West Georgia,Houston - GA,GA,WEST_GEORGIA_101_JESSICA_DR_31093,00094A 041000,00094A 041000,SFR,,Pending,11/03/2026,,3,2,26-8251,1196,SALE_PENDING
21,"3754 Ga Highway 120, Tallapoosa, GA 30176",West Georgia,Haralson - GA,GA,WEST_GEORGIA_3754_GA_HIGHWAY_120_30176,0061 A 0015,0061 A 0015,SFR,,Pending,11/03/2026,209089,3,1,15-3107F12,1184,SALE_PENDING
22,"217 Beatrice Ct Ne, Ranger, GA 30734",West Georgia,Gordon - GA,GA,WEST_GEORGIA_217_BEATRICE_CT_NE_30734,095A 084,095A 084,SFR,,Pending,11/03/2026,,3,2,26-003227-01,1216,SALE_PENDING
24,"103 Pine Ave Sw, Griffin, GA 30224",West Georgia,Spalding - GA,GA,WEST_GEORGIA_103_PINE_AVE_SW_30224,142 04017,142 04017,SFR,,Pending,11/03/2026,198097,2,1,26-8233,1326,SALE_PENDING


Auction Status
Pending      554
Cancelled    148
Killed        37
Active        19
Name: count, dtype: int64


In [40]:
create_auction.head(5)


,Listing Type,Property Address,Property City,oldCounty,State,Zip,notStatus,Auction Date,Property Type,Bldg Sq Ft,...,Salesforce Online Bidding,Lead Manager Name,Lead Manager,Tax ID,APN,Type of Property,Auction Opening Bid,Auction Status,Killed Reason,Auction Date Updated
5,Foreclosure,101B OAK HILL DRIVE,CALHOUN,Gordon,GA,30701,SALE_PENDING,2026-11-03,Single family home,1216,...,NaN,David Riehle,0051H00000ASDES,,WEST_GEORGIA_101B_OAK_HILL_DRIVE_30701,SFR,,Pending,,11/03/2026
7,Foreclosure,185 ROCK BRIDGE DR,RINGGOLD,Catoosa,GA,30736,SALE_PENDING,2026-11-03,Single family home,1351,...,NaN,David Riehle,0051H00000ASDES,,WEST_GEORGIA_185_ROCK_BRIDGE_DR_30736,SFR,,Pending,,11/03/2026
10,Foreclosure,272 GRAND MAGNOLIA STREET,JACKSON,Butts,GA,30233,SALE_PENDING,2026-11-03,Single family home,1778,...,NaN,Scott Autenreith,0051H000009cubT,0033C-064-000.,EAST_GEORGIA_272_GRAND_MAGNOLIA_STREET_30233,SFR,,Pending,,11/03/2026
14,Foreclosure,1145 COREY CREEK DR,MONROE,Walton,GA,30655,CANCELED,2026-11-03,Single family home,1899,...,NaN,Scott Autenreith,0051H000009cubT,N166E069,EAST_GEORGIA_1145_COREY_CREEK_DR_30655,SFR,,Cancelled,,11/03/2026
16,Foreclosure,2840 GA HWY 127,KATHLEEN,Houston,GA,31047,SALE_PENDING,2026-11-03,Single family home,1202,...,NaN,David Riehle,0051H00000ASDES,00128B031000,WEST_GEORGIA_2840_GA_HWY_127_31047,SFR,,Pending,,11/03/2026


In [41]:
create_auction.to_csv(
    export_data / 'create_auction.csv',
    index=False
)